# LiDAR workflow (TERN STAC API)
Find a point-cloud asset, and read it.


## Before you run
- Update placeholder values (`COLLECTION_ID`, dates, bounds, point coordinates) to match your data.
- Ensure auth is configured for protected assets (for example `.netrc` and/or GDAL config).
- Install optional dependencies required by this notebook's workflow (`odc-stac`, `rioxarray`, `geopandas`, plotting extras).
- Run cells from top to bottom so variables are initialized in order.


In [ ]:
from pathlib import Path

from tern_stac import TernStacClient, get_item_asset_href, laz_to_canopy_height, preview_raster


In [ ]:
# Fill in from your catalog values
# Find the collection ID, item ID, and asset key from stac api browser
# TODO: remove "-test" from the URL when using production catalog
# https://stac-api-test.tern.org.au/stac-browser/?.language=en
COLLECTION_ID = "uas__dronescape_lidar"
POINT_CLOUD_ITEM_ID = "20251123-SAAEYB0029.copc"
POINT_CLOUD_ASSET_KEY = "20251123-SAAEYB0029.copc.laz"
POINT_CLOUD_MEDIA_TYPE = None
POINT_CLOUD_ROLE = None


In [ ]:
Path("outputs").mkdir(exist_ok=True)

client = TernStacClient()
search = client.search(collections=[COLLECTION_ID])
items = list(search.items())
len(items)


In [ ]:
item = None
for candidate in items:
    if candidate.id == POINT_CLOUD_ITEM_ID:
        item = candidate
        break

if item is None:
    raise RuntimeError("Item ID not found in results. Set POINT_CLOUD_ITEM_ID to a valid item id.")

asset_href = get_item_asset_href(
    item,
    asset_key=POINT_CLOUD_ASSET_KEY if POINT_CLOUD_ASSET_KEY != "<asset_key>" else None,
    media_type=POINT_CLOUD_MEDIA_TYPE,
    role=POINT_CLOUD_ROLE,
)
asset_href


In [ ]:
import laspy
with laspy.CopcReader.open(asset_href) as crdr:
    try:
        points = crdr.read_points()
        las = laspy.LasData(header=crdr.header, points=points)
    except Exception:
        points = crdr.query()
        las = laspy.lasdata.LasData(header=crdr.header, points=points)
las


In [ ]:
# NOTE: this example workflow was using an improper CHM function that was not actually producing a CHM.
# The function has been removed from the public API and this example notebook is now for demonstration purposes only.
# Go to reference of `laz_to_canopy_height` in this notebook for more details, or if you really want to try it.

In [ ]:
# (DEPRECATED) 
# chm = laz_to_canopy_height(
#     asset_href,
#     resolution=1.0,
#     save_tif="outputs/drone_lidar_chm.tif",
# )
# chm


In [ ]:
# preview_raster(
#     chm,
#     cmap="viridis",
#     title="Canopy height model",
#     save_path="outputs/drone_lidar_chm_preview.png",
# )
